# 15 · The Copilot harness – J.A.R.V.I.S. stops writing his own loop 🧪

> 🧪 **Preview.** The Microsoft Copilot harness is a public preview (announced in [the Foundry blog](https://devblogs.microsoft.com/foundry/microsoft-copilot-harness-in-foundry/)). Names, fields and limits may change. This notebook uses `AIProjectClient(..., allow_preview=True)` and `azure-ai-projects>=2.9.0`.

Until now Tony built every loop himself: the function-call loop in chapter 05, a whole container in chapter 09. The **Copilot harness** is a managed runtime for **prompt agents**. Foundry runs the agent loop and a sandbox with a workspace. You still only write configuration: model, instructions, tools and **skills**. Chapter 07 told you that a skill reference on a default prompt agent has no effect; on the harness it works. And a skill is more than text: `invoice-review` ships five **scripts**, and the harness runs them in its sandbox.

| | Default prompt agent | Prompt agent + Copilot harness 🧪 | Hosted agent |
|---|---|---|---|
| You write | definition | definition (`harness=MicrosoftCopilotHarness()`) | code |
| Agent loop | Foundry | Foundry (the harness) | your code |
| Skills (`skills=[SkillReference]`) | ❌ accepted, no effect | ✅ | your code loads them |
| Skill scripts | ❌ | ✅ run with the shell tool | your code runs them |
| Built-in tools | none | file read, file write, shell, web fetch, subagents | whatever you code |
| Workspace (files between turns) | ❌ | ✅ one per conversation | sessions |
| Deploy | `create_version` | `create_version` or `azd deploy` | `azd deploy` |

> ⚠️ **Before you start.** The project region must support the harness (Sweden Central, this repo's default, does), and the model must support reasoning and tool calling. You choose the harness when you **create** an agent and cannot switch it later. The harness runtime is free until 2026-12-01; after that the sandbox is billed per vCPU-hour and GiB-hour, on top of model tokens.

Deep dive: [docs/copilot-harness.md](../docs/copilot-harness.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import extract_invoice      # Document Intelligence prebuilt-invoice (chapter 13)
from stark_harness import (                  # this chapter's helpers, read them in notebooks/stark_harness.py
    publish_skill, ask_harness, follows_escalation, invoice_request, review_invoice, show_ledger,
)

import json
from importlib.metadata import version
from azure.ai.projects.models import (
    MicrosoftCopilotHarness, MicrosoftCopilotToolsetConfig, MicrosoftCopilotToolsetPreview,
    PromptAgentDefinition, SkillReference,
)

print("azure-ai-projects", version("azure-ai-projects"))   # needs 2.9.0 or later for MicrosoftCopilotHarness
preview_project = project(preview=True)                       # allow_preview=True: skills and the harness are preview
SDK_AGENT_NAME = "jarvis-nb15"      # created here with the SDK
AZD_AGENT_NAME = "jarvis-harness"   # deployed with azd in section 6 (name from src/jarvis-harness/azure.yaml)

## 1 · Publish the skills as packages
The harness reads skills from Foundry, not from disk. A skill is a **folder**: `SKILL.md` plus anything it needs. Chapter 07 uploaded only the text (`inline_content`). Here we zip each folder and upload it with `create_from_files`, so the scripts and data files travel with the instructions:

```
skills/invoice-review/
├── SKILL.md                 checklist + the exact commands to run
├── scripts/                 check_invoice.py, vendor_check.py, iban_check.py, ledger.py, approval_limit.py
└── assets/                  vendors.csv, approval-limits.csv, invoice-example.json
```

`publish_skill()` in [stark_harness.py](stark_harness.py) zips the folder with `SKILL.md` at the root of the archive and makes one call:

```python
preview_project.beta.skills.create_from_files(name, content=CreateSkillVersionFromFilesBody(
    files=[(f"{name}.zip", zip_bytes, "application/zip")]))
```

Each upload is a new immutable version. We keep the version numbers to pin them on the agent.

In [ ]:
skill_versions = {s.name: s.version for s in (publish_skill("stark-escalation"), publish_skill("invoice-review"))}
print(skill_versions)

## 2 · Create an agent on the harness
Three things differ from chapter 04:

| Field | Value | Why |
|---|---|---|
| `harness` | `MicrosoftCopilotHarness()` | selects the managed runtime. Fixed for the life of the agent |
| `skills` | `SkillReference(name, version)` | attaches the skills. Each agent version pins one skill version |
| `tools` | one `MicrosoftCopilotToolsetPreview` | switches built-in harness tools on or off. All five are on by default |

This time **shell stays on**: it is how the agent runs the skill's scripts. Shell is the broadest tool (it can read, write and reach the network on its own), so we switch off what J.A.R.V.I.S. does not need: `subagents`. To also limit where the sandbox may connect, attach a network egress policy (preview, not done here, see [docs/copilot-harness.md](../docs/copilot-harness.md)). The harness manages its own settings, so leave out `temperature`, `top_p`, `tool_choice` and `text.format`.

In [ ]:
definition = PromptAgentDefinition(
    model=model("chat"),
    instructions=("You are J.A.R.V.I.S., Tony Stark's assistant in the workshop. "
                  "Use your skills when a request matches them, and run a skill's scripts when it ships them. "
                  "Keep answers short."),
    harness=MicrosoftCopilotHarness(),
    skills=[SkillReference(name=n, version=v) for n, v in skill_versions.items()],
    tools=[MicrosoftCopilotToolsetPreview(configs=[
        MicrosoftCopilotToolsetConfig(name="shell", enabled=True),       # runs the invoice-review scripts
        MicrosoftCopilotToolsetConfig(name="subagents", enabled=False),
    ])],
)
sdk_agent = preview_project.agents.create_version(agent_name=SDK_AGENT_NAME, definition=definition)
print(sdk_agent.name, "version", sdk_agent.version)

# what Foundry persisted, as plain JSON
stored = preview_project.agents.get(agent_name=SDK_AGENT_NAME).versions.latest.definition.as_dict()
print(json.dumps({k: stored.get(k) for k in ("kind", "harness", "skills", "tools")}, indent=2))

## 3 · Ask it, and check that the skill is used
Two rules apply to every harness request:

- every request needs a **conversation** (no `previous_response_id`, no `store=False`)
- use an **agent-bound client** (`get_openai_client(agent_name=...)`), like the Microsoft Learn sample

`ask_harness()` in [stark_harness.py](stark_harness.py) follows both rules:

```python
oai = preview_project.get_openai_client(agent_name=agent_name)
response = oai.responses.create(conversation=conversation_id, input=text)
```

`stark-escalation` leaves a marker, as in chapter 07: the answer starts with `Severity:`. The harness often sends a short preamble message first ("I'll use the escalation skill."), so `follows_escalation()` checks the final message, the answer itself. The output items show what the harness did (a `skill` call, file tools, shell, reasoning).

In [ ]:
INCIDENT_QUESTION = "A suit just started smoking in the hangar. What do I do?"

sdk_client = preview_project.get_openai_client(agent_name=SDK_AGENT_NAME)
sdk_conversation = sdk_client.conversations.create()
conversations_to_delete = [(SDK_AGENT_NAME, sdk_conversation.id)]

incident_reply = ask_harness(SDK_AGENT_NAME, sdk_conversation.id, INCIDENT_QUESTION)
print("follows stark-escalation:", follows_escalation(incident_reply))

## 4 · A skill that runs scripts
A prompt is good at judgement and bad at arithmetic, checksums and lookups. `invoice-review` therefore ships **prewritten commands**. [SKILL.md](../skills/invoice-review/SKILL.md) tells the agent which one to run for which check. The model only fills in the arguments:

| Check | Command | Fails when |
|---|---|---|
| fields, arithmetic, date | `python scripts/check_invoice.py invoice.json` | a required field is missing, the lines or the total don't add up, the date is in the future |
| vendor master | `python scripts/vendor_check.py "<vendor>"` | the vendor is blocked or unknown ([vendors.csv](../skills/invoice-review/assets/vendors.csv)) |
| bank account | `python scripts/iban_check.py "<IBAN>" --vendor "<vendor>"` | the mod-97 checksum is wrong, or the IBAN is not the one on file |
| duplicate | `python scripts/ledger.py invoice.json` | the vendor's invoice number is already in `ledger.csv` (in the workspace) |
| approval limit | `python scripts/approval_limit.py --approver "<name>" --total <total>` | the total is above the approver's limit ([approval-limits.csv](../skills/invoice-review/assets/approval-limits.csv)) |

Every script prints one JSON line and exits with `0` (pass) or `1` (fail). Standard library only, so they need nothing installed. They are plain CLI tools, so try them here first, on your machine:

In [ ]:
S = "skills/invoice-review/scripts"
sh(f"python {S}/check_invoice.py skills/invoice-review/assets/invoice-example.json")
sh(f'python {S}/vendor_check.py "Oscorp Materials"')
sh(f'python {S}/iban_check.py "GB82 WEST 1234 5698 7654 32" --vendor "Pym Particle Supply Co."')
sh(f'python {S}/approval_limit.py --approver "Happy Hogan" --total 5726.34')

Now the same commands, run by the agent in its sandbox. **Who does what:**

- This notebook reads the PDFs with **Document Intelligence** (`extract_invoice` from chapter 13, with your identity) and sends the extracted fields as JSON.
- The agent saves them as `invoice.json` in its workspace and runs the scripts with the shell tool.

> ❓ The sandbox calling Document Intelligence itself, with the **agent identity**, is not documented for scripts: Foundry uses the agent identity for tools (toolbox, OpenAPI, file upload), and Microsoft Learn shows no way for a sandbox script to get a token. So extraction stays outside the sandbox.

Four cases follow, each built to trip a different script. Run them **in order**: case A opens a new conversation, `invoice_conversation`, and cases B to D continue in it, because case C needs the ledger that case A leaves in its workspace. To repeat the cases, start again at case A.

First, extract the two sample invoices:

In [ ]:
pym_invoice = extract_invoice("data/docs/invoice-pym-4471.pdf")
oscorp_invoice = extract_invoice("data/docs/invoice-oscorp-9930.pdf")
for name, inv in [("pym", pym_invoice), ("oscorp", oscorp_invoice)]:
    print(name, inv["fields"]["VendorName"], inv["fields"]["InvoiceId"], inv["fields"]["InvoiceTotal"])

Every case sends the agent a plain-text **prompt**: who approves, optionally the IBAN and a message from the vendor's email, and the fields that Document Intelligence extracted. Nothing else: no script names, no checklist. Which checks to run comes from the skill. For case A the prompt looks like this (shortened):

```text
Review this invoice. Approver: Pepper Potts.
Payment IBAN from the vendor's email: DE89 3704 0044 0532 0130 00
Fields extracted by Document Intelligence:
{
  "fields": {
    "VendorName": "Pym Particle Supply Co.",
    "InvoiceId": "4471",
    "InvoiceTotal": 3332.34,
    ...
  },
  "confidence": { "VendorName": 0.91, "InvoiceTotal": 0.44, ... },
  "items": [ ... ]
}
```

`invoice_request()` builds that text, so each case cell shows exactly which inputs go in. `review_invoice()` (both in [stark_harness.py](stark_harness.py)) sends it and shows the turn as formatted Markdown, in three parts:

| Part | Shows |
|---|---|
| **Prompt** | the text lines sent to the agent. The extracted JSON is folded under **Extracted fields**: click to open it |
| **Answer** | all of the agent's messages, one paragraph each (often a short preamble, then the answer with its reasons), rendered as Markdown |
| **Check** | the verdict next to the expected one, the skill scripts named in the output items, and the scripts the case needs but the agent did not run (`missing`) |

The model decides what to run, so a run can differ. If a case is off, open its trace (section 👀). In a test run, shell commands came back as `shell_call` items and the file tools as `function_call` items (for example `view` on `/home/session/workspace/ledger.csv`). ❓ Where the harness places the skill's own files in the sandbox was not checked, so `scripts_run()` simply searches all output items for `scripts/<name>.py`.

In [ ]:
ALL_SCRIPTS = {"check_invoice", "vendor_check", "iban_check", "ledger", "approval_limit"}
PYM_IBAN_ON_FILE = "DE89 3704 0044 0532 0130 00"     # the account on file in vendors.csv

### Case A · A clean invoice

**Goal:** the baseline. A correct invoice from an approved vendor, paid to the account on file, so every check has something to pass. It also writes invoice 4471 to the workspace ledger, which case C needs.

Case A starts a **new conversation**, so its workspace and `ledger.csv` start empty. Sent twice in the same conversation, case A would be a duplicate of itself and get `REJECT`.

| Input | Value |
|---|---|
| invoice | `invoice-pym-4471.pdf`, as extracted |
| approver | Pepper Potts (limit 250,000) |
| IBAN | `DE89 3704 0044 0532 0130 00`, the one in `vendors.csv` |

**Expected:** `APPROVE`. All five scripts run and pass: arithmetic, vendor, IBAN, ledger and approval limit.

> 💡 Document Intelligence may read the total with **low confidence** (0.44 in one test run, the threshold is 0.8). `check_invoice.py` doesn't escalate it: the line items, subtotal and tax add up to exactly that total, so it lists the total in `confirmed_by_arithmetic` instead of `warnings`. A rule in code settles what the model would otherwise send to a human.

In [ ]:
invoice_conversation = sdk_client.conversations.create()     # new conversation: new, empty workspace
conversations_to_delete.append((SDK_AGENT_NAME, invoice_conversation.id))

case_a_prompt = invoice_request(pym_invoice, approver="Pepper Potts", iban=PYM_IBAN_ON_FILE)
review_invoice(SDK_AGENT_NAME, invoice_conversation.id, case_a_prompt, expected="APPROVE", must_run=ALL_SCRIPTS)

### Case B · A blocked vendor

**Goal:** a fact the model cannot know. The Oscorp invoice is correct on paper, so a model that only reads it finds nothing wrong. The vendor master says otherwise.

| Input | Value |
|---|---|
| invoice | `invoice-oscorp-9930.pdf`, as extracted |
| approver | Pepper Potts |
| IBAN | none given |

**Expected:** `REJECT`, because `vendor_check.py` fails. The reason quotes the note from [vendors.csv](../skills/invoice-review/assets/vendors.csv): Oscorp is under investigation for industrial espionage. Other scripts may run too; only `vendor_check` must.

In [ ]:
case_b_prompt = invoice_request(oscorp_invoice, approver="Pepper Potts")
review_invoice(SDK_AGENT_NAME, invoice_conversation.id, case_b_prompt, expected="REJECT", must_run={"vendor_check"})

### Case C · "We changed banks"

**Goal:** the classic payment-diversion fraud. The vendor resends invoice 4471 and asks to pay a new account. The new IBAN has a valid mod-97 checksum, so it looks fine to a model. Two scripts catch it, because they compare against data instead of judging the text.

| Input | Value |
|---|---|
| invoice | `invoice-pym-4471.pdf` again, the same one as case A |
| approver | Pepper Potts |
| IBAN | `GB82 WEST 1234 5698 7654 32`, valid but not on file |
| vendor message | "We changed banks, please pay this invoice to our new account." |

**Expected:** `REJECT`, for two reasons:

- `iban_check.py`: the IBAN is valid, but it is not the account on file for Pym.
- `ledger.py`: invoice 4471 is already in `ledger.csv`, written in case A. Case A ran in this conversation, so the entry is there.

In [ ]:
case_c_prompt = invoice_request(pym_invoice, approver="Pepper Potts", iban="GB82 WEST 1234 5698 7654 32",
                                vendor_message="We changed banks, please pay this invoice to our new account.")
review_invoice(SDK_AGENT_NAME, invoice_conversation.id, case_c_prompt, expected="REJECT", must_run={"iban_check", "ledger"})

### Case D · A typo in the total

**Goal:** arithmetic, where models are weakest. Invoice 4502 is typed in by hand, with two digits swapped in the total: 4,885.50 + 390.84 is 5,276.34, but the invoice says 5,726.34. A model reading the JSON can easily miss it.

| Input | Value |
|---|---|
| invoice | Pym 4502, typed in below (no PDF) |
| approver | Happy Hogan (no own row, so the `default` limit of 5,000) |
| IBAN | none given |

**Expected:** `REJECT`, because `check_invoice.py` fails and prints the exact numbers. `approval_limit.py` would also flag it: 5,726.34 is above Happy's limit.

In [ ]:
typo_invoice = {"fields": {"VendorName": "Pym Particle Supply Co.", "InvoiceId": "4502", "InvoiceDate": "2026-10-20",
                   "SubTotal": 4885.50, "TotalTax": 390.84, "InvoiceTotal": 5726.34},
        "items": [{"description": "Shrink/grow regulator", "quantity": 4, "unitprice": 1200.0, "amount": 4800.0},
                  {"description": "Shipping and handling", "quantity": 1, "unitprice": 85.5, "amount": 85.5}]}
case_d_prompt = invoice_request(typo_invoice, approver="Happy Hogan")
review_invoice(SDK_AGENT_NAME, invoice_conversation.id, case_d_prompt, expected="REJECT", must_run={"check_invoice"})

## 5 · The workspace: files that live as long as the conversation
Each conversation on the harness gets a **workspace**: a folder in a Foundry-managed sandbox that survives between turns. `ledger.py` keeps its `ledger.csv` there, which is how case C knew that invoice 4471 had been seen before. Three steps show where that file lives and how long: the same question in two conversations, then the same invoice.

### Step 1 · Look into the workspace

`show_ledger()` ([stark_harness.py](stark_harness.py)) asks the agent for its `ledger.csv` and checks for invoice 4471.

**Goal:** see that the scripts left a real file behind, in `invoice_conversation`, the conversation of cases A to D.

**Expected:** the agent shows `ledger.csv` with a row for Pym invoice 4471 from case A. Rows for Oscorp 9930 and Pym 4502 appear only if the agent also ran `ledger.py` in cases B and D; the skill doesn't require it once a check has failed.

In [ ]:
show_ledger(SDK_AGENT_NAME, invoice_conversation.id)

### Step 2 · The same question in a new conversation

**Goal:** show that a workspace belongs to one conversation. Same agent, same question, only the conversation is new.

**Expected:** the agent finds no `ledger.csv`: the new workspace is empty. The check says `lists invoice 4471: no`. Run this step again before step 3 if you repeat it: step 3 writes a ledger into `fresh_conversation`.

In [ ]:
fresh_conversation = sdk_client.conversations.create()
conversations_to_delete.append((SDK_AGENT_NAME, fresh_conversation.id))
show_ledger(SDK_AGENT_NAME, fresh_conversation.id)

### Step 3 · The same invoice in the new conversation

**Goal:** see what the empty workspace means for a review. We send `case_a_prompt` again, unchanged, to `fresh_conversation`.

| | Case C (section 4) | This step |
|---|---|---|
| conversation | `invoice_conversation` | `fresh_conversation` |
| `ledger.csv` before the review | contains 4471 from case A | does not exist (step 2) |
| `ledger.py` | fails: duplicate | passes, and writes a new `ledger.csv` |

**Expected:** `APPROVE`, with all five scripts run, exactly as in case A. The invoice that was a duplicate a moment ago is new again here.

In [ ]:
review_invoice(SDK_AGENT_NAME, fresh_conversation.id, case_a_prompt, expected="APPROVE", must_run=ALL_SCRIPTS)

The workspace is **session state, not storage**: Foundry may idle or replace the sandbox, and the workspace is gone with the conversation. A real ledger belongs in a database the scripts can reach. To hand a file back to the user, the harness uploads it to Foundry Files with the **agent identity**, which first needs the **Foundry User** role on the project (not done here, see [docs/copilot-harness.md](../docs/copilot-harness.md)).

## 6 · The same agent as a file, deployed with azd
In the code-first world the agent lives next to the skills it uses. [src/jarvis-harness/azure.yaml](../src/jarvis-harness/azure.yaml) declares:

| Service | Host | Does |
|---|---|---|
| `ai-project` | `azure.ai.project` | points at the STARK project (`STARK_PROJECT_ENDPOINT`), creates nothing |
| `stark-escalation`, `invoice-review` | `azure.ai.skill` | publish the folder `skills/<name>` (scripts and assets included) as a new skill version (`archive:`) |
| `jarvis-harness` | `azure.ai.agent`, `kind: prompt` | the agent: `harness.type: microsoft_copilot_preview`, `skills:`, the toolset with shell on |

It is its **own azd project**: the harness needs `azd` 1.37.0+, `azure.ai.agents` 1.0.0-beta.22+ and the opt-in flag `AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true`, while the root [azure.yaml](../azure.yaml) pins older extensions for the hosted agents.

In [ ]:
print((ROOT / "src" / "jarvis-harness" / "azure.yaml").read_text())

Run once in a terminal, or let the next cell do it:

```bash
azd extension upgrade microsoft.foundry          # azure.ai.agents >= 1.0.0-beta.22
python scripts/harness_env.py                    # azd env `stark-harness` in src/jarvis-harness, filled from .env
cd src/jarvis-harness
export AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true    # PowerShell: $env:AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS = "true"
azd deploy --all --no-prompt                     # predeploy hook copies skills/, then: skills, then the agent
azd ai agent invoke --new-conversation --no-prompt "A suit just started smoking in the hangar. What do I do?"
```

> ⚠️ After the upgrade, the root `azure.yaml` (which pins `azure.ai.agents` `=1.0.0-beta.20`) may refuse to run. To go back for chapters 08, 09 and 16, reinstall that version: `azd extension install azure.ai.agents --version 1.0.0-beta.20 --force`.

In [ ]:
import os
os.environ["AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS"] = "true"   # the azd opt-in; inherited by sh()

sh("azd ai agent version")
sh("python scripts/harness_env.py")
sh("cd src/jarvis-harness && azd deploy --all --no-prompt")

Check what Foundry stored: `kind: prompt`, `harness.type: microsoft_copilot_preview`, two skill references with **pinned** versions, and the toolset.

In [ ]:
deployed = preview_project.agents.get(agent_name=AZD_AGENT_NAME).versions.latest
d = deployed.definition.as_dict()   # model objects -> plain JSON
print("version", deployed.version)
print(json.dumps({k: d.get(k) for k in ("kind", "harness", "skills", "tools")}, indent=2))

## 7 · Call the deployed agent
Same client, same rules: an agent-bound client and a conversation. Invoice A runs all five scripts again, in the deployed agent's own sandbox.

In [ ]:
azd_client = preview_project.get_openai_client(agent_name=AZD_AGENT_NAME)
azd_conversation = azd_client.conversations.create()
conversations_to_delete.append((AZD_AGENT_NAME, azd_conversation.id))
incident_reply = ask_harness(AZD_AGENT_NAME, azd_conversation.id, INCIDENT_QUESTION)
print("follows stark-escalation:", follows_escalation(incident_reply))
review_invoice(AZD_AGENT_NAME, azd_conversation.id, case_a_prompt, expected="APPROVE", must_run=ALL_SCRIPTS)   # case A

## 👀 In the portal
- **Build → Agents**: `jarvis-nb15` and `jarvis-harness`. The agent page shows the harness, **Harness tools** (subagents off) and the two skills.
- Open a response and select **Traces**: the tool calls include `skill`, then the shell commands `python .../scripts/*.py` with their JSON output.
- ❓ Where the portal lists skills on their own (**Tools → Skills (Preview)** in the Microsoft Learn docs) and whether it shows a package's files was not checked.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Deleting a conversation also ends its harness session and workspace. Individual `responses.delete` is not supported on the harness. The skills stay: chapter 07 created `stark-escalation` first, and keeping them costs nothing.

In [ ]:
for agent_name, cid in conversations_to_delete:
    try: preview_project.get_openai_client(agent_name=agent_name).conversations.delete(cid)
    except Exception as exc: print("skip conversation", cid, type(exc).__name__)
delete_agent(SDK_AGENT_NAME)
delete_agent(AZD_AGENT_NAME)      # or, in src/jarvis-harness: azd ai agent delete --force --no-prompt

## 🎯 Recap
The Copilot harness turns a prompt agent into an agent with a managed loop, a workspace, built-in tools and **skills that work**, without a container. A skill can carry **scripts**: the agent runs the prewritten commands with the shell tool, and the deterministic checks (arithmetic, checksums, lookups, duplicates) no longer depend on the model's maths. You choose the harness at creation, give every request a conversation and switch off the built-in tools you do not need. The same agent fits in an `azure.yaml` next to its skills, so `azd deploy --all` ships both.

| Need | Pick |
|---|---|
| instructions + Foundry tools are enough | default prompt agent (chapter 04) |
| skills with scripts, files, multi-step tool work, no code to host | prompt agent + Copilot harness (this chapter) 🧪 |
| your own framework, loop or dependencies | hosted agent (chapter 09) |

To compare, `jarvis-skills` in [09 · Hosted agent](09_hosted_agent.ipynb) runs the same skills with a loop you own. Before chapter 16, go back to `azure.ai.agents` `1.0.0-beta.20` if you upgraded in section 6.

> 💡 Next: Act VII – the Avengers. [16 · Workflows](16_workflows.ipynb).